# Embedding Model Training Notebook

This notebook mirrors `scripts/train_model_with_embedding.py` and runs end-to-end training/inference.

In [1]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.datasets import PAD_TOKEN_ID, UNK_TOKEN_ID, build_city_dataloaders
from src.features import (
    build_booker_device_vocabs,
    build_city_sequence_pack,
    build_city_vocab,
    create_multiple_sequences,
)
from src.models import CityGRU, CityTransformer
from src.training.embedding import recommend_top4_cities, train_embedding_model
from src.utils import data_dir, print_accuracy_at_4_report, submission_dir, top_city_ids_from_train

In [3]:
# ---- Config (edit as needed) ----
EPOCHS = 5
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
MULTI_STEP = False
TOPK_CANDIDATES = 50
MODEL_TYPE = "transformer"  # "transformer" or "gru"
POOLING = "last"            # Transformer only: "last" / "mean" / "cls"
SKIP_EVAL = False
GROUND_TRUTH_PATH = None  # e.g. str(data_dir() / "ground_truth.csv")

# ---- Data ----
out_dir = submission_dir()
out_dir.mkdir(parents=True, exist_ok=True)

train_set = pd.read_csv(data_dir() / "train_set.csv")
test_set = pd.read_csv(data_dir() / "test_set.csv")

print("正在聚合行程序列...")
train_trips = create_multiple_sequences(train_set)
test_trips = create_multiple_sequences(test_set)

city_to_idx, idx_to_city = build_city_vocab(train_set)
vocab_size = len(city_to_idx) + 2

booker_to_idx, device_to_idx, n_booker, n_device = build_booker_device_vocabs(train_trips)

train_pack = build_city_sequence_pack(
    train_trips,
    city_to_idx,
    is_test=False,
    multi_step=MULTI_STEP,
    booker_to_idx=booker_to_idx,
    device_to_idx=device_to_idx,
)
test_pack = build_city_sequence_pack(
    test_trips,
    city_to_idx,
    is_test=True,
    multi_step=False,
    booker_to_idx=booker_to_idx,
    device_to_idx=device_to_idx,
)

print(
    f"✅ City 数据集完成！训练样本: {len(train_pack.x)} | 测试样本: {len(test_pack.x)} "
    f"| multi_step={MULTI_STEP} | model={MODEL_TYPE} | pooling={POOLING}"
)

train_ctx = (
    train_pack.ctx_booker,
    train_pack.ctx_device,
    train_pack.ctx_month,
    train_pack.ctx_stay,
    train_pack.ctx_trip_len,
    train_pack.ctx_num_unique_cities,
    train_pack.ctx_repeat_city_ratio,
    train_pack.ctx_last_stay_days,
    train_pack.ctx_same_country_streak,
)
test_ctx = (
    test_pack.ctx_booker,
    test_pack.ctx_device,
    test_pack.ctx_month,
    test_pack.ctx_stay,
    test_pack.ctx_trip_len,
    test_pack.ctx_num_unique_cities,
    test_pack.ctx_repeat_city_ratio,
    test_pack.ctx_last_stay_days,
    test_pack.ctx_same_country_streak,
)
train_loader, test_loader = build_city_dataloaders(
    train_pack.x,
    train_pack.y,
    test_pack.x,
    batch_size=BATCH_SIZE,
    train_ctx=train_ctx,
    test_ctx=test_ctx,
)

正在聚合行程序列...
✅ City 数据集完成！训练样本: 217573 | 测试样本: 70662 | multi_step=False | model=transformer | pooling=last


In [7]:
import torch

# 取一个 batch
batch = next(iter(train_loader))

print("batch fields:", len(batch))
print("batch_x shape:", batch[0].shape)
print("batch_y shape:", batch[1].shape)

# 拿第1条样本
batch_x = batch[0]
batch_y = batch[1]

x0 = batch_x[0]   # 序列（已padding）
y0 = batch_y[0]   # 标签（下一个城市token）

print("x0 (token ids):", x0.tolist())
print("y0 token:", int(y0.item()))

# 去掉PAD看看真实前缀（PAD_TOKEN_ID=0）
x0_nopad = x0[x0 != PAD_TOKEN_ID]
print("x0 no PAD:", x0_nopad.tolist())

# token -> city_id（只对 >=2 的普通token有效）
prefix_city_ids = [idx_to_city.get(int(t), None) for t in x0_nopad.tolist() if int(t) >= 2]
target_city_id = idx_to_city.get(int(y0.item()), None) if int(y0.item()) >= 2 else None

print("prefix city_ids:", prefix_city_ids)
print("target city_id:", target_city_id)

batch fields: 11
batch_x shape: torch.Size([256, 14])
batch_y shape: torch.Size([256])
x0 (token ids): [17605, 27412, 4399, 7079, 35639, 18409, 26592, 39778, 0, 0, 0, 0, 0, 0]
y0 token: 23537
x0 no PAD: [17605, 27412, 4399, 7079, 35639, 18409, 26592, 39778]
prefix city_ids: [29770, 46258, 7358, 11877, 60274, 31146, 44869, 67353]
target city_id: 39820


In [ ]:
# ---- Model + Train ----
if MODEL_TYPE == "transformer":
    model = CityTransformer(
        vocab_size=vocab_size,
        pad_token_id=PAD_TOKEN_ID,
        d_model=256,
        nhead=4,
        num_layers=2,
        n_booker_countries=n_booker,
        n_device_classes=n_device,
        pooling=POOLING,
    )
elif MODEL_TYPE == "gru":
    model = CityGRU(
        vocab_size=vocab_size,
        pad_token_id=PAD_TOKEN_ID,
        embedding_dim=256,
        hidden_dim=256,
        n_booker_countries=n_booker,
        n_device_classes=n_device,
    )
else:
    raise ValueError(f"Unsupported MODEL_TYPE: {MODEL_TYPE}")

model = train_embedding_model(
    model,
    train_loader,
    pad_token_id=PAD_TOKEN_ID,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING,
)

# ---- Predict + Save ----
top_popular = top_city_ids_from_train(train_set, k=4)
predictions = recommend_top4_cities(
    model=model,
    test_loader=test_loader,
    idx_to_city=idx_to_city,
    top_popular_cities=top_popular,
    reserved_token_ids={PAD_TOKEN_ID, UNK_TOKEN_ID},
    topk_candidates=TOPK_CANDIDATES,
)

submission_df = pd.DataFrame(
    predictions,
    columns=["city_id_1", "city_id_2", "city_id_3", "city_id_4"],
)
submission_df.insert(0, "utrip_id", test_trips["utrip_id"].tolist())

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = out_dir / f"submission_embedding_{timestamp}.csv"
submission_df.to_csv(submission_path, index=False)
print(f"✅ {submission_path} 已生成！")

print_accuracy_at_4_report(
    submission_df,
    skip=SKIP_EVAL,
    ground_truth_path=GROUND_TRUTH_PATH,
)

submission_df.head()